In [1]:
import pandas as pd

df = pd.read_table("data1.txt")



In [6]:

df.head()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13240 entries, 0 to 13239
Data columns (total 27 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   time              13240 non-null  object 
 1   DeviceName        13240 non-null  object 
 2   AccX(g)           13240 non-null  float64
 3   AccY(g)           13240 non-null  float64
 4   AccZ(g)           13240 non-null  float64
 5   AsX(°/s)          13240 non-null  float64
 6   AsY(°/s)          13240 non-null  float64
 7   AsZ(°/s)          13240 non-null  float64
 8   AngleX(°)         13240 non-null  float64
 9   AngleY(°)         13240 non-null  float64
 10  AngleZ(°)         13240 non-null  float64
 11  HX(uT)            13240 non-null  float64
 12  HY(uT)            13240 non-null  float64
 13  HZ(uT)            13240 non-null  float64
 14  TrajectoryX(mm)   13240 non-null  int64  
 15  TrajectoryY(mm)   13240 non-null  int64  
 16  TrajectoryZ(mm)   13240 non-null  int64 

In [7]:
df.head(5)

,time,DeviceName,AccX(g),AccY(g),AccZ(g),AsX(°/s),AsY(°/s),AsZ(°/s),AngleX(°),AngleY(°),...,SpeedX(mm/s),SpeedY(mm/s),SpeedY(mm/s).1,Q0(),Q1(),Q2(),Q3(),Temperature(°C),Version(),Battery level(%)
0,2026-3-2 18:15:40.748,WT901BLE67(D94B4858716D),0.82,3.732,-2.261,209.167,645.264,-574.28,-105.36,71.17,...,530,132,-68,0.09100,0.57971,0.35138,-0.72943,22.2,10080.1.21,100
1,2026-3-2 18:15:40.786,WT901BLE67(5FFE38C0499E),0.00,0.000,1.027,0.000,0.000,0.00,-90.34,12.55,...,-121,209,-50,-0.70139,0.69632,-0.15176,0.00531,22.3,10080.1.21,100
2,2026-3-2 18:15:40.837,WT901BLE67(D94B4858716D),0.82,3.732,-2.261,209.167,645.264,-574.28,-106.30,71.89,...,-18,-117,-156,-0.10513,0.72943,0.09464,-0.66919,22.2,10080.1.21,100
3,2026-3-2 18:15:40.847,WT901BLE67(5FFE38C0499E),0.00,0.000,1.027,0.000,0.000,0.00,-92.48,10.21,...,-366,589,-161,-0.70139,0.69632,-0.15176,0.00531,22.3,10080.1.21,100
4,2026-3-2 18:15:40.929,WT901BLE67(D94B4858716D),0.82,3.732,-2.261,209.167,645.264,-574.28,-112.09,75.18,...,4,-64,-43,-0.10513,0.72943,0.09464,-0.66919,22.2,10080.1.21,100


In [8]:
df["DeviceName"].value_counts()

DeviceName
WT901BLE67(5FFE38C0499E)    10194
WT901BLE67(D94B4858716D)     3046
Name: count, dtype: int64

In [10]:
speed_cols = [c for c in df.columns if "Speed" in c]
print(speed_cols)


['SpeedX(mm/s)', 'SpeedY(mm/s)', 'SpeedY(mm/s).1']


In [11]:
df[["SpeedX(mm/s)", "SpeedY(mm/s)", "SpeedY(mm/s).1"]].describe()



,SpeedX(mm/s),SpeedY(mm/s),SpeedY(mm/s).1
count,13240.000000,13240.000000,13240.000000
mean,-632.715634,-1447.567296,-280.666012
std,15909.313434,16914.794164,16161.722049
min,-32767.000000,-32762.000000,-32753.000000
25%,-11131.250000,-13333.000000,-10784.250000
50%,-748.000000,-2218.000000,-656.000000
75%,9173.750000,9569.750000,10947.750000
max,32745.000000,32763.000000,32760.000000


In [12]:
df[["SpeedX(mm/s)", "SpeedY(mm/s)", "SpeedY(mm/s).1"]].corr()

,SpeedX(mm/s),SpeedY(mm/s),SpeedY(mm/s).1
SpeedX(mm/s),1.000000,0.010057,-0.024817
SpeedY(mm/s),0.010057,1.000000,-0.002817
SpeedY(mm/s).1,-0.024817,-0.002817,1.000000


In [13]:
df = df.rename(columns={
    "SpeedY(mm/s).1": "SpeedZ(mm/s)"
})

In [14]:
import numpy as np

right = df[df["DeviceName"]=="WT901BLE67(D94B4858716D)"].copy()
right = right.sort_values("time")
right["time"] = pd.to_datetime(right["time"])

t = right["time"].astype("int64") / 1e9  # seconds

vx = right["SpeedX(mm/s)"] / 1000
vy = right["SpeedY(mm/s)"] / 1000
vz = right["SpeedZ(mm/s)"] / 1000

v = np.sqrt(vx**2 + vy**2 + vz**2)

dt = np.diff(t, prepend=t.iloc[0])
dt[0] = 0

distance = np.sum(v * dt)

print("Distance (meters):", distance)

Distance (meters): 1222.2374068234992


In [16]:

left = df[df["DeviceName"]=="WT901BLE67(5FFE38C0499E)"].copy()


left["time"] = pd.to_datetime(left["time"])


left = left.sort_values("time")


t = left["time"].astype("int64") / 1e9


vx = left["SpeedX(mm/s)"] / 1000
vy = left["SpeedY(mm/s)"] / 1000
vz = left["SpeedZ(mm/s)"] / 1000

v = np.sqrt(vx**2 + vy**2 + vz**2)

dt = np.diff(t, prepend=t.iloc[0])
dt[0] = 0

dist_left = np.sum(v * dt)

print("Left foot distance (meters):", dist_left)

Left foot distance (meters): 30870.824914150446


In [17]:
print("Right samples:", len(right))
print("Left samples:", len(left))

print("\nRight time range:", right["time"].min(), "→", right["time"].max())
print("Left time range :", left["time"].min(), "→", left["time"].max())

Right samples: 3046
Left samples: 10194

Right time range: 2026-03-02 18:15:40.748000 → 2026-03-02 18:20:47.580000
Left time range : 2026-03-02 18:15:40.786000 → 2026-03-02 18:32:47.540000


In [18]:
# Make sure both are datetime (you already did this, but safe)
right["time"] = pd.to_datetime(right["time"])
left["time"]  = pd.to_datetime(left["time"])

start = right["time"].min()
end   = right["time"].max()

left_trim = left[(left["time"] >= start) & (left["time"] <= end)].copy()

print("Left before:", len(left))
print("Left after :", len(left_trim))
print("Trimmed left time range:", left_trim["time"].min(), "->", left_trim["time"].max())

Left before: 10194
Left after : 3046
Trimmed left time range: 2026-03-02 18:15:40.786000 -> 2026-03-02 18:20:47.572000


In [19]:
t = left_trim["time"].astype("int64") / 1e9

vx = left_trim["SpeedX(mm/s)"] / 1000
vy = left_trim["SpeedY(mm/s)"] / 1000
vz = left_trim["SpeedZ(mm/s)"] / 1000

v = np.sqrt(vx**2 + vy**2 + vz**2)

dt = np.diff(t, prepend=t.iloc[0])
dt[0] = 0

dist_left_trim = np.sum(v * dt)
print("Left foot distance (trimmed, meters):", dist_left_trim)

Left foot distance (trimmed, meters): 9226.551665155826


In [20]:
def speed_stats(d, label):
    vx = d["SpeedX(mm/s)"].to_numpy()/1000
    vy = d["SpeedY(mm/s)"].to_numpy()/1000
    vz = d["SpeedZ(mm/s)"].to_numpy()/1000
    v  = (vx*vx + vy*vy + vz*vz) ** 0.5
    print(label, "v median:", np.median(v), "m/s",
          "| v 95%:", np.percentile(v,95), "m/s",
          "| v max:", np.max(v), "m/s")

speed_stats(right, "RIGHT")
speed_stats(left_trim, "LEFT ")

RIGHT v median: 5.191768348539882 m/s | v 95%: 17.700396210386227 m/s | v max: 34.37767020901795 m/s
LEFT  v median: 32.673213296492015 m/s | v 95%: 45.12004642320892 m/s | v max: 54.50204671386204 m/s


In [21]:
import numpy as np
import pandas as pd

def _to_seconds(dt_series: pd.Series) -> np.ndarray:
    """Convert datetime series to seconds (float)."""
    return (dt_series.astype("int64") / 1e9).to_numpy()

def _speed_magnitude_mps(d: pd.DataFrame, speed_cols) -> np.ndarray:
    """Return speed magnitude in m/s from mm/s columns."""
    sx, sy, sz = speed_cols
    vx = d[sx].to_numpy(dtype=float) / 1000.0
    vy = d[sy].to_numpy(dtype=float) / 1000.0
    vz = d[sz].to_numpy(dtype=float) / 1000.0
    return np.sqrt(vx * vx + vy * vy + vz * vz)

def _dt_stats_seconds(t_sec: np.ndarray) -> dict:
    """Compute dt stats in seconds from time array in seconds."""
    if len(t_sec) < 2:
        return {"n": len(t_sec), "median": np.nan, "p95": np.nan, "max": np.nan}
    dt = np.diff(t_sec)
    return {
        "n": len(dt),
        "median": float(np.median(dt)),
        "p95": float(np.percentile(dt, 95)),
        "max": float(np.max(dt)),
    }

def _v_stats(v: np.ndarray) -> dict:
    if len(v) == 0:
        return {"median": np.nan, "p95": np.nan, "max": np.nan}
    return {
        "median": float(np.median(v)),
        "p95": float(np.percentile(v, 95)),
        "max": float(np.max(v)),
    }

def _print_section(title: str):
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))

def profile_imu_dataset(
    df: pd.DataFrame,
    time_col: str = "time",
    device_col: str = "DeviceName",
    speed_cols=("SpeedX(mm/s)", "SpeedY(mm/s)", "SpeedZ(mm/s)"),
):
    """
    Print a reliability profile for a multi-sensor IMU dataset.
    This version prints only (no automatic pass/fail), but is structured
    so you can easily add thresholds later.
    """

    # --- Basic checks ---
    required = [time_col, device_col, *speed_cols]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    d0 = df.copy()
    d0[time_col] = pd.to_datetime(d0[time_col], errors="coerce")
    if d0[time_col].isna().any():
        n_bad = int(d0[time_col].isna().sum())
        print(f"WARNING: {n_bad} rows have invalid '{time_col}' after parsing.")

    # Split by device
    devices = list(d0[device_col].dropna().unique())
    if len(devices) == 0:
        raise ValueError(f"No devices found in column '{device_col}'.")

    _print_section("DATASET OVERVIEW")
    print("Rows:", len(d0))
    print("Devices:", len(devices))
    for dev in devices:
        print(" -", dev)

    # --- Per-device stats ---
    per = {}  # store computed stats for later comparison

    _print_section("PER-DEVICE STATS")
    for dev in devices:
        d = d0[d0[device_col] == dev].copy()
        d = d.sort_values(time_col)

        tmin, tmax = d[time_col].min(), d[time_col].max()
        duration_s = (tmax - tmin).total_seconds() if pd.notna(tmin) and pd.notna(tmax) else np.nan

        t_sec = _to_seconds(d[time_col]) if pd.notna(tmin) else np.array([])
        dt_s = _dt_stats_seconds(t_sec)

        v = _speed_magnitude_mps(d, speed_cols) if len(d) else np.array([])
        vs = _v_stats(v)

        per[dev] = {
            "n": len(d),
            "tmin": tmin,
            "tmax": tmax,
            "duration_s": duration_s,
            "dt": dt_s,
            "v": vs,
        }

        print(f"\nDevice: {dev}")
        print(f"  samples        : {len(d)}")
        print(f"  time range     : {tmin} -> {tmax}")
        print(f"  duration (min) : {duration_s/60:.2f}" if np.isfinite(duration_s) else "  duration (min) : NaN")
        print(f"  dt median/p95/max (s): {dt_s['median']:.4f} / {dt_s['p95']:.4f} / {dt_s['max']:.4f}")
        print(f"  speed median/p95/max (m/s): {vs['median']:.3f} / {vs['p95']:.3f} / {vs['max']:.3f}")

        # "Flags" (prints only)
        flags = []
        if len(d) < 1000:
            flags.append("LOW_SAMPLE_COUNT")
        if np.isfinite(duration_s) and duration_s < 30:
            flags.append("VERY_SHORT_RECORDING")
        if np.isfinite(dt_s["max"]) and dt_s["max"] > 1.0:
            flags.append("BIG_TIME_GAPS (dt_max > 1s)")
        if np.isfinite(vs["p95"]) and vs["p95"] > 10:
            flags.append("VERY_HIGH_SPEEDS (p95 > 10 m/s)")
        if np.isfinite(vs["max"]) and vs["max"] > 20:
            flags.append("EXTREME_SPEED_SPIKES (max > 20 m/s)")

        print("  flags          :", (", ".join(flags) if flags else "None"))

    # --- Cross-device comparison ---
    if len(devices) >= 2:
        _print_section("CROSS-DEVICE COMPARISON (FIRST TWO DEVICES)")

        a, b = devices[0], devices[1]
        a_min, a_max = per[a]["tmin"], per[a]["tmax"]
        b_min, b_max = per[b]["tmin"], per[b]["tmax"]

        overlap_start = max(a_min, b_min)
        overlap_end = min(a_max, b_max)
        overlap_s = (overlap_end - overlap_start).total_seconds()

        print(f"Devices compared :\n  A = {a}\n  B = {b}")
        print(f"Overlap window   : {overlap_start} -> {overlap_end}")
        print(f"Overlap duration : {overlap_s/60:.2f} min")

        # Compute speed scale ratio using overlap window (median + p95)
        if overlap_s > 0:
            da = d0[(d0[device_col] == a) & (d0[time_col] >= overlap_start) & (d0[time_col] <= overlap_end)].copy()
            db = d0[(d0[device_col] == b) & (d0[time_col] >= overlap_start) & (d0[time_col] <= overlap_end)].copy()

            da = da.sort_values(time_col)
            db = db.sort_values(time_col)

            va = _speed_magnitude_mps(da, speed_cols)
            vb = _speed_magnitude_mps(db, speed_cols)

            if len(va) and len(vb):
                ratio_med = np.median(vb) / max(np.median(va), 1e-9)
                ratio_p95 = np.percentile(vb, 95) / max(np.percentile(va, 95), 1e-9)
                print(f"Speed ratio B/A (median): {ratio_med:.3f}")
                print(f"Speed ratio B/A (p95)   : {ratio_p95:.3f}")

                flags = []
                if ratio_med > 2.0 or ratio_med < 0.5:
                    flags.append("POSSIBLE_SCALE_MISMATCH (median ratio far from 1)")
                if abs(ratio_med - ratio_p95) > 0.5:
                    flags.append("INCONSISTENT_RATIO (not a simple constant scale)")

                print("Comparison flags :", (", ".join(flags) if flags else "None"))
            else:
                print("Could not compute overlap ratios (one device has no overlap samples).")
        else:
            print("No time overlap between devices, cannot compare directly.")

    _print_section("NEXT STEPS")
    print(
        "You can turn the printed 'flags' into automatic validity decisions by adding\n"
        "threshold parameters (e.g., vmax_p95, dt_max_allowed, min_samples, max_ratio).\n"
        "For now this gives you a consistent, readable report per dataset."
    )

In [22]:
profile_imu_dataset(df)


DATASET OVERVIEW
Rows: 13240
Devices: 2
 - WT901BLE67(D94B4858716D)
 - WT901BLE67(5FFE38C0499E)

PER-DEVICE STATS

Device: WT901BLE67(D94B4858716D)
  samples        : 3046
  time range     : 2026-03-02 18:15:40.748000 -> 2026-03-02 18:20:47.580000
  duration (min) : 5.11
  dt median/p95/max (s): 0.0900 / 0.2538 / 0.6650
  speed median/p95/max (m/s): 5.192 / 17.700 / 34.378
  flags          : VERY_HIGH_SPEEDS (p95 > 10 m/s), EXTREME_SPEED_SPIKES (max > 20 m/s)

Device: WT901BLE67(5FFE38C0499E)
  samples        : 10194
  time range     : 2026-03-02 18:15:40.786000 -> 2026-03-02 18:32:47.540000
  duration (min) : 17.11
  dt median/p95/max (s): 0.0900 / 0.2390 / 0.7200
  speed median/p95/max (m/s): 32.049 / 44.531 / 56.136
  flags          : VERY_HIGH_SPEEDS (p95 > 10 m/s), EXTREME_SPEED_SPIKES (max > 20 m/s)

CROSS-DEVICE COMPARISON (FIRST TWO DEVICES)
Devices compared :
  A = WT901BLE67(D94B4858716D)
  B = WT901BLE67(5FFE38C0499E)
Overlap window   : 2026-03-02 18:15:40.786000 -> 2026-03

In [23]:
#HSR Tme Function (>6km/h)

import numpy as np
import pandas as pd

def hsr_time(d, threshold_kmh=6.0):
    """
    Returns total High Speed Running time (seconds)
    where speed > threshold_kmh.
    """

    d = d.sort_values("time").copy()
    d["time"] = pd.to_datetime(d["time"])

    # Convert speed from mm/s to m/s
    vx = d["SpeedX(mm/s)"].to_numpy() / 1000
    vy = d["SpeedY(mm/s)"].to_numpy() / 1000
    vz = d["SpeedZ(mm/s)"].to_numpy() / 1000

    # Speed magnitude (m/s)
    v = np.sqrt(vx**2 + vy**2 + vz**2)

    # Convert to km/h
    v_kmh = v * 3.6

    # Compute dt
    t = d["time"].astype("int64") / 1e9
    dt = np.diff(t, prepend=t.iloc[0])
    dt[0] = 0

    # Mask for HSR
    mask = v_kmh > threshold_kmh

    # Total HSR time in seconds
    return float(np.sum(dt[mask]))

right_hsr_time = hsr_time(right)
left_hsr_time  = hsr_time(left_trim)

print("Right HSR time (seconds):", right_hsr_time)
print("Left  HSR time (seconds):", left_hsr_time)

print("Right HSR time (minutes):", right_hsr_time / 60)
print("Left  HSR time (minutes):", left_hsr_time / 60)

Right HSR time (seconds): 253.06099843978882
Left  HSR time (seconds): 299.5189998149872
Right HSR time (minutes): 4.217683307329813
Left  HSR time (minutes): 4.991983330249786


In [24]:
#LSR Time Function (<6 kh/h)

import numpy as np
import pandas as pd

def lsr_time(d, threshold_kmh=6.0):
    """
    Returns total Low Speed Running time (seconds)
    where speed < threshold_kmh.
    """

    d = d.sort_values("time").copy()
    d["time"] = pd.to_datetime(d["time"])

    # Convert speed from mm/s to m/s
    vx = d["SpeedX(mm/s)"].to_numpy() / 1000
    vy = d["SpeedY(mm/s)"].to_numpy() / 1000
    vz = d["SpeedZ(mm/s)"].to_numpy() / 1000

    # Speed magnitude (m/s)
    v = np.sqrt(vx**2 + vy**2 + vz**2)

    # Convert to km/h
    v_kmh = v * 3.6

    # Compute dt
    t = d["time"].astype("int64") / 1e9
    dt = np.diff(t, prepend=t.iloc[0])
    dt[0] = 0

    # Mask for LSR
    mask = v_kmh < threshold_kmh

    # Total LSR time in seconds
    return float(np.sum(dt[mask]))

right_lsr_time = lsr_time(right)
left_lsr_time  = lsr_time(left_trim)

print("Right LSR time (minutes):", right_lsr_time / 60)
print("Left  LSR time (minutes):", left_lsr_time / 60)

Right LSR time (minutes): 0.8961833596229554
Left  LSR time (minutes): 0.12111667394638062


In [ ]:
#Distance covered in HSR

import numpy as np
import pandas as pd

def hsr_distance(d, threshold_kmh=6.0):
    """
    Returns total distance (meters) covered
    when speed > threshold_kmh.
    """

    d = d.sort_values("time").copy()
    d["time"] = pd.to_datetime(d["time"])

    # Convert speed from mm/s to m/s
    vx = d["SpeedX(mm/s)"].to_numpy() / 1000
    vy = d["SpeedY(mm/s)"].to_numpy() / 1000
    vz = d["SpeedZ(mm/s)"].to_numpy() / 1000

    # Speed magnitude (m/s)
    v = np.sqrt(vx**2 + vy**2 + vz**2)

    # Convert to km/h for threshold comparison
    v_kmh = v * 3.6

    # Compute dt
    t = d["time"].astype("int64") / 1e9
    dt = np.diff(t, prepend=t.iloc[0])
    dt[0] = 0

    # Mask for HSR
    mask = v_kmh > threshold_kmh

    # Distance in meters
    return float(np.sum(v[mask] * dt[mask]))

right_hsr_dist = hsr_distance(right)
left_hsr_dist  = hsr_distance(left_trim)

print("Right HSR distance (m):", right_hsr_dist)
print("Left  HSR distance (m):", left_hsr_dist)

print("Right HSR distance (km):", right_hsr_dist / 1000)
print("Left  HSR distance (km):", left_hsr_dist / 1000)

Right HSR distance (m): 2062.9817347316502
Left  HSR distance (m): 9217.85007577829
Right HSR distance (km): 2.0629817347316504
Left  HSR distance (km): 9.217850075778289


In [26]:
#Distance Covered LSR

import numpy as np
import pandas as pd

def lsr_distance(d, threshold_kmh=6.0):
    """
    Returns total distance (meters) covered
    when speed < threshold_kmh.
    """

    d = d.sort_values("time").copy()
    d["time"] = pd.to_datetime(d["time"])

    # Convert speed from mm/s to m/s
    vx = d["SpeedX(mm/s)"].to_numpy() / 1000
    vy = d["SpeedY(mm/s)"].to_numpy() / 1000
    vz = d["SpeedZ(mm/s)"].to_numpy() / 1000

    # Speed magnitude (m/s)
    v = np.sqrt(vx**2 + vy**2 + vz**2)

    # Convert to km/h for threshold comparison
    v_kmh = v * 3.6

    # Compute dt
    t = d["time"].astype("int64") / 1e9
    dt = np.diff(t, prepend=t.iloc[0])
    dt[0] = 0

    # Mask for LSR
    mask = v_kmh < threshold_kmh

    # Distance in meters
    return float(np.sum(v[mask] * dt[mask]))

right_lsr_dist = lsr_distance(right)
left_lsr_dist  = lsr_distance(left_trim)

print("Right LSR distance (m):", right_lsr_dist)
print("Left  LSR distance (m):", left_lsr_dist)

print("Right LSR distance (km):", right_lsr_dist / 1000)
print("Left  LSR distance (km):", left_lsr_dist / 1000)

Right LSR distance (m): 34.05574603099703
Left  LSR distance (m): 3.7747984276849946
Right LSR distance (km): 0.03405574603099703
Left  LSR distance (km): 0.003774798427684995
